In [1]:
from qutip import *
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import HTML
from tqdm.notebook import trange
import cProfile, pstats, io
from pstats import SortKey
import warnings
import sys
sys.path.append( '../src' )
from QFINumerics import QFI, genChannel, searchForState, calc_for_state

/home/dominic/software-projects/lab-repos/covert-quantum-analysis/notebooks/../libs/QFINumerics.py:4: TqdmExperimentalWarning: Using `tqdm.autonotebook.tqdm` in notebook mode. Use `tqdm.tqdm` instead to force console mode (e.g. in jupyter console)
  from tqdm.autonotebook import trange


In [ ]:
state = searchForState()

  0%|          | 0/20000 [00:00<?, ?it/s]

In [ ]:
n = tensor(num(5),identity(5),identity(6))
chan = genChannel()
pth0 = thermal_dm(5,(1-.1)*.1)
rho = vector_to_operator(chan*operator_to_vector(state.proj()))
rel_ent = entropy_relative(pth0,rho.ptrace([0]),tol=1e-50)
max_QFI = QFI(rho,n)
print(rel_ent)
print(max_QFI)

In [ ]:
def calc_for_state(rho,n,As,args=[12,.1,.1]):
    # Input
    #  - rho: the initial state. The first field should be the signal field,
    #         all ancilla fields aren't impacted by our procedure
    #  - n: the number operator for the signal field
    #  - As: the annihilation operator for the signal field
    #  - args: an array of arguments that define our setup
    #    - 0: the number of dimensions to use for the thermal
    #         fields truncated Fock space
    #    - 1: the average photon number in the thermal field
    #    - 2: the transmisivity of our beamsplitter
    # Output
    #  - rel_ent: the quantum relative entropy between our state after mixing
    #             with thermal background, and the field without our input state
    #  - QFI: the quantum Fisher information of our state after mixing with
    #         thermal background
    # Description: Simulates a signal field mixed with a thermal background,
    #              and gives a measure of the effectiveness of a measurement
    #              and the detectability of our measurement to an adversary

    # Here we extract our parameters from our argument
    thdims = args[0]
    nth = args[1]
    eta = args[2]
    theta = np.arccos(np.sqrt(eta))
    
    # Since the thermal mixing is phase invariant, we don't need to worry about
    # when we pass through the sample. Putting the sample before or after the mixing
    # yields identical results. Additionally since phase transformations are unitary
    # we don't need to worry about applying it at all.
    pth = thermal_dm(thdims,nth)
    pth0 = thermal_dm(rho.dims[0][0],(1-eta)*nth)
    at = destroy(thdims)
    G = theta*(tensor(at.dag(),As)+tensor(at,As.dag()))*1j
    U = G.expm()
    st = tensor(pth, rho)
    stm = U*st*U.dag()

    # Tracing out the thermal mode should just involve doing a partial trace over
    # just the thermal mode. Unfortunately, QuTiP only exposes a function that
    # traces over all **but** the specified modes. Therefore, we just specify
    # all but the first mode
    tokeep = np.arange(1,len(stm.dims[0]))
    rhom = stm.ptrace(tokeep)

    # The adversary only has access to the signal mode when trying to determine
    # if we are making a measurement, so we trace over all other modes
    rel_ent = entropy_relative(pth0,rhom.ptrace([0]),tol=1e-50)
    
    FI = QFI(rhom,n)
    return [rel_ent,FI]

In [ ]:
dim = 5
nth = .1
a = destroy(dim)
ad = a.dag()
iden = identity(dim)
pth = thermal_dm(dim,nth)
n = tensor(num(dim),iden)
a1 = tensor(a,iden)
theta = np.pi/2
zeroket = basis(dim,0)
oneket = basis(dim,1)
twoket = basis(dim,2)

In [ ]:
QFIs = np.zeros(dim)
REs = np.zeros(dim)
nth = .1
eta = .1
for i in range(dim):
    nket = basis(dim,i)
    unnorm = tensor(zeroket,nket) + tensor(nket,zeroket)
    statet = unnorm.proj().unit()
    [REs[i],QFIs[i]] = calc_for_state(statet,n,a1,[dim,nth,eta])
fig,ax = plt.subplots(2)
ax[0].set_xlabel('Input Fock state')
ax[1].set_xlabel('Input Fock state')
ax[1].set_ylabel('Relative Entropy')
ax[1].set_ylabel('QFI')
ax[0].set_title(r'Relative Entropy vs Fock number $n_b$ = {nb}, $\eta$ = {eta}'.format(nb=nth,eta=eta))
ax[0].set_title(r'QFI vs Fock number $n_b$ = {nb}, $\eta$ = {eta}'.format(nb=nth,eta=eta))
ax[0].plot(REs)
ax[1].plot(QFIs)

In [ ]:
QFIs = np.zeros(4)
REs = np.zeros(4)
nth = .1
eta = .1
xsi = .13
S = squeezing(tensor(a,iden),tensor(iden,a),xsi)
sq = S*tensor(zeroket,zeroket)
for i in range(4):
    unnorm = a1**i * sq
    statet = unnorm.unit().proj()
    [REs[i],QFIs[i]] = calc_for_state(statet,n,a1,[dim,nth,eta])
fig,ax = plt.subplots(2)
ax[0].set_xlabel('Input Fock state')
ax[1].set_xlabel('Input Fock state')
ax[0].set_ylabel('Relative Entropy')
ax[1].set_ylabel('QFI')
ax[0].set_title(r'QFI & Relative Entropy vs Fock number $n_b$ = {nb}, $\eta$ = {eta}, $n_s$ = {nsq}'.format(nb=nth,eta=eta,nsq = np.sinh(eta/2)**2))
ax[0].plot(REs)
ax[1].plot(QFIs)

In [ ]:
states = []
wdims = 6
sdims = 5
idims = 5
probabilities = np.zeros(wdims)
for i in range(wdims):
    staten = tensor(identity(sdims),identity(idims),basis(wdims,i).dag())*state
    probabilities[i] = staten.norm()**2
    toadd = qutip.dimensions.from_tensor_rep(np.reshape(qutip.dimensions.to_tensor_rep(staten.unit()),[sdims,idims]),[[sdims],[idims]])
    states.append(toadd)
print(probabilities)
[fig,ani] = anim_matrix_histogram(states,bar_style='abs',color_style='phase',options={'bars_alpha':.8})
HTML(ani.to_jshtml())

In [ ]:
states = []
wdims = 6
sdims = 5
idims = 5
probabilities = np.zeros(wdims)
n = tensor(num(sdims),identity(idims))
chan = genChannel()
rho = vector_to_operator(chan*operator_to_vector(state.proj()))
QFIs = np.zeros(wdims)
for i in range(wdims):
    staten = (tensor(identity(sdims),identity(idims),basis(wdims,i).dag())*rho*tensor(identity(sdims),identity(idims),basis(wdims,i))).drop_scalar_dims(inplace=True)
    probabilities[i] = staten.norm()
    QFIs[i] = QFI(staten.unit(),n)
print(probabilities)
print(QFIs)
print(np.sum(probabilities*QFIs))

In [ ]:
adverstate = state.ptrace([0])
nbar = expect(num(sdims),adverstate)
print(nbar)
thermeq = thermal_dm(sdims,nbar)
entropy_relative(thermeq,adverstate)

In [ ]:
plot_wigner(adverstate,projection='3d')

In [ ]:
plot_wigner(thermeq,projection='3d')

In [ ]:
plot_wigner(rand_ket(5))